# EXP29 — Procrustes on naturalistic factual recall (TriviaQA, Gemma-2 9B -> 2B)
#
# STANDALONE. Run cells 1-2 once, RESTART KERNEL, then run 2 to the end.
#
# The question: your factual-recall arm finds that the reconstruction map matches the task
# map, and explains it by saying naturalistic answers are distinctive enough to survive a
# reconstructive map. Procrustes preserves strictly MORE of the donor state than the
# reconstruction map does, so that explanation predicts Procrustes should confer at least as
# well. This tests that prediction.
#
# Everything reuses EXP27b's pipeline verbatim: same facts.jsonl, same 60/40 split with the
# same shuffle seed, same strict answer matching, same generation-defined unsolvable bin.
# If the bin size and the recon-map numbers reproduce your published values, the task-map
# numbers already in exp27_results.json sit on the SAME bin and are directly comparable.

In [1]:
!pip install -q "transformers==4.46.3" accelerate datasets numpy huggingface_hub hf_transfer
!pip uninstall -y torchvision torchaudio
print("deps installed — RESTART THE KERNEL now, then continue from CELL 2")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124
deps installed — RESTART THE KERNEL now, then continue from CELL 2


In [4]:
# === CELL 2: imports + RUN CONFIG ===========================================
import os, json, math, random, time, gc, re
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
from transformers.utils import is_torch_available

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert is_torch_available(), (
    f"transformers {transformers.__version__} does not see torch {torch.__version__}. "
    "Re-run cell 1 and RESTART THE KERNEL.")
print(f"torch {torch.__version__} | transformers {transformers.__version__} | cuda {torch.cuda.is_available()}")

MODEL_D, MODEL_R = "google/gemma-2-9b", "google/gemma-2-2b"
L_D, L_R = 34, 20                      # the paper's validated pair

# Upload the SAME facts file the published EXP27b run used. "facts (1).jsonl" and
# "facts (3).jsonl" are byte-identical (sha256 bee764e4...), 40,000 rows, and either is
# correct. A plain 8,000-row "facts.jsonl" is the OLDER v1 file: it gives a 471-item bin,
# not the 570 in your paper, and nothing would be comparable. Cell 7 refuses it.
FACT_SOURCE_CANDIDATES = ["facts (3).jsonl", "facts (1).jsonl", "facts.jsonl"]
FACT_EXPECT_ROWS = 40000               # the published run encoded exactly this many
FACT_TRAIN_FRAC = 0.6                  # identical to EXP27b
FACT_MAX_EVAL   = 3000                 # identical to EXP27b
MAX_NEW_FACT    = 12                   # identical to EXP27b
BATCH           = 16
RIDGE_LAMBDA    = 1e3

RUN_PROBE = False   # optional: fit an answer probe on each map's output. Not needed for the
                    # comparison; turn on only if you want to look inside the maps.

OUT_JSON = "exp29_facts_procrustes.json"
RESULTS = {"_config": dict(donor=MODEL_D, recipient=MODEL_R, L_D=L_D, L_R=L_R,
                           train_frac=FACT_TRAIN_FRAC,
                           max_eval=FACT_MAX_EVAL, max_new=MAX_NEW_FACT)}
def save():
    with open(OUT_JSON, "w") as f: json.dump(RESULTS, f, indent=2)
    print(f"  [saved {OUT_JSON}]")
print(f"{MODEL_D} (L{L_D}) -> {MODEL_R} (L{L_R})")

torch 2.4.1+cu124 | transformers 4.46.3 | cuda True
google/gemma-2-9b (L34) -> google/gemma-2-2b (L20)


In [ ]:
# === CELL 3: Hugging Face login =============================================
from huggingface_hub import login
HF_TOKEN = ""
login(HF_TOKEN)
print("logged in")

logged in


In [6]:
# === CELL 4: statistics helpers (identical to the eval notebooks) ===========
BOOT_B = 2000
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=BOOT_B, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x)
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"
def mcnemar_exact(a, b):
    from fractions import Fraction
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01 = int((~a & b).sum()); n10 = int((a & ~b).sum()); n = n01 + n10
    if n == 0: return dict(n01=0, n10=0, p=1.0)
    k = min(n01, n10)
    tot = Fraction(0)
    for i in range(0, k+1): tot += Fraction(math.comb(n, i))
    return dict(n01=n01, n10=n10, p=min(1.0, float(2*tot/Fraction(2)**n)))
print("stats ready")

stats ready


In [7]:
# === CELL 5: helpers — verbatim from EXP27b so the bin is identical =========
def _hid(o):  return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W

_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]: return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

def _norm(t):
    return "".join(c for c in t.lower() if c.isascii() and (c.isalnum() or c == " ")).strip()

def _match(gen_text, gold):
    """Strict: normalized equality, or gold followed by a word boundary.
    Identical to EXP27b -- do not loosen, 'Au' must not match 'Australia'."""
    g, p = _norm(gold), _norm(gen_text.split("\n")[0])
    return bool(g) and (p == g or p.startswith(g + " "))

def _first_answer_token(tok, prompt, answer):
    p = tok(prompt).input_ids
    f = tok(prompt + " " + answer).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and tok.decode([f[j]]).strip() == "": j += 1
    return (torch.tensor(f[:j]), f[j]) if j < len(f) else (None, None)

@torch.inference_mode()
def states_last_and_top(model, layer, items, batch=BATCH):
    acc, top = [], []
    for i in range(0, len(items), batch):
        ids, m = left_pad([p["ids"] for p in items[i:i+batch]], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        acc.append(out.hidden_states[layer+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return torch.cat(acc), top

@torch.inference_mode()
def _generate(model, items, vec_fn=None, layer=None, batch=BATCH):
    outs, h = [], None
    if vec_fn is not None:
        h = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(items), batch):
            chunk = items[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            if vec_fn is not None:
                v = vec_fn(i, len(chunk))
                assert v.shape[0] == len(chunk), f"graft {v.shape[0]} != {len(chunk)}"
                _graft["vec"] = v
            gen = model.generate(ids.to(DEVICE), attention_mask=m.to(DEVICE),
                                 max_new_tokens=MAX_NEW_FACT, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
            outs += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
    finally:
        if h: h.remove()
        _graft["vec"] = None
    return outs
print("helpers ready")

helpers ready


In [8]:
# === CELL 6: load both models ===============================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_R)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model_r = AutoModelForCausalLM.from_pretrained(
    MODEL_R, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
model_d = AutoModelForCausalLM.from_pretrained(
    MODEL_D, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
with torch.inference_mode():
    _hl = model_d(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0,-1,:]
assert not torch.isnan(_hl).any(), "donor produced NaN logits"
del _hl
D_D, D_R = model_d.config.hidden_size, model_r.config.hidden_size
print(f"loaded | d_donor={D_D} d_recipient={D_R}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/2.38G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

loaded | d_donor=3584 d_recipient=2304


In [9]:
# === CELL 7: load facts, split, build the unsolvable bin ====================
import hashlib
FACT_SOURCE = next((f for f in FACT_SOURCE_CANDIDATES if os.path.exists(f)), None)
assert FACT_SOURCE is not None, (
    "No facts file found. Upload one of: " + ", ".join(FACT_SOURCE_CANDIDATES))
_rows = [l for l in open(FACT_SOURCE) if l.strip()]
_sha = hashlib.sha256(open(FACT_SOURCE, "rb").read()).hexdigest()
print(f"using {FACT_SOURCE}  ({len(_rows)} rows, sha256 {_sha[:16]})")
assert len(_rows) == FACT_EXPECT_ROWS, (
    f"{FACT_SOURCE} has {len(_rows)} rows, expected {FACT_EXPECT_ROWS}. An 8,000-row file is "
    "the OLDER v1 fact set -- it produces a 471-item bin, not the 570 in the paper, so none "
    "of the published task-map numbers would be comparable. Upload the 40,000-row file.")
RESULTS["_config"]["fact_source"] = FACT_SOURCE
RESULTS["_config"]["fact_source_sha256"] = _sha
RESULTS["_config"]["fact_source_rows"] = len(_rows)
_raw = [json.loads(l) for l in _rows]
FACTS = []
for r in _raw:
    ids, tid = _first_answer_token(tokenizer, r["prompt"], r["answer"])
    if tid is not None:
        FACTS.append(dict(prompt=r["prompt"], answer=r["answer"], ids=ids, tok=tid))
print(f"{len(_raw)} candidates -> {len(FACTS)} encoded")

_fi = list(range(len(FACTS))); random.Random(0).shuffle(_fi)     # SAME seed as EXP27b
_cut = max(1, int(len(_fi) * FACT_TRAIN_FRAC))
FT_TRAIN = [FACTS[i] for i in _fi[:_cut]]
FT_EVAL  = [FACTS[i] for i in _fi[_cut:]][:FACT_MAX_EVAL]
print(f"train={len(FT_TRAIN)}  eval={len(FT_EVAL)}")

t0 = time.time()
print("building the bin by generation from both models (the slow part) ...")
_d_txt = _generate(model_d, FT_EVAL)
_r_txt = _generate(model_r, FT_EVAL)
F_donor_ok = [_match(_d_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
F_recip_ok = [_match(_r_txt[i], FT_EVAL[i]["answer"]) for i in range(len(FT_EVAL))]
F_UNSOLV = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and not F_recip_ok[i]]
F_SOLV   = [i for i in range(len(FT_EVAL)) if F_donor_ok[i] and F_recip_ok[i]]
print(f"donor {sum(F_donor_ok)}/{len(FT_EVAL)} = {sum(F_donor_ok)/len(FT_EVAL):.3f} | "
      f"recipient {sum(F_recip_ok)}/{len(FT_EVAL)} = {sum(F_recip_ok)/len(FT_EVAL):.3f}")
print(f">>> UNSOLVABLE BIN n = {len(F_UNSOLV)}   [{time.time()-t0:.0f}s]")
print(">>> EXP27b published n = 570, donor 0.659, recipient 0.502.")
print(">>> If these do not match, the pipeline diverged and the task-map numbers in")
print(">>> exp27_results.json are NOT comparable to what this notebook produces.")
RESULTS["bins"] = dict(n_encoded=len(FACTS), n_train=len(FT_TRAIN), n_eval=len(FT_EVAL),
                       n_unsolvable=len(F_UNSOLV), n_solvable=len(F_SOLV),
                       donor_solve_rate=round(sum(F_donor_ok)/len(FT_EVAL), 4),
                       recipient_solve_rate=round(sum(F_recip_ok)/len(FT_EVAL), 4),
                       _published_reference=dict(n_unsolvable=570, donor=0.659, recipient=0.502))
assert len(F_UNSOLV) >= 50, "bin too small"
save()

using facts (3).jsonl  (40000 rows, sha256 bee764e4c5012ead)
40000 candidates -> 40000 encoded
train=24000  eval=3000
building the bin by generation from both models (the slow part) ...
donor 1977/3000 = 0.659 | recipient 1501/3000 = 0.500
>>> UNSOLVABLE BIN n = 571   [219s]
>>> EXP27b published n = 570, donor 0.659, recipient 0.502.
>>> If these do not match, the pipeline diverged and the task-map numbers in
>>> exp27_results.json are NOT comparable to what this notebook produces.
  [saved exp29_facts_procrustes.json]


In [10]:
# === CELL 8: states, reconstruction map, Procrustes map =====================
t0 = time.time()
FX9t, _ = states_last_and_top(model_d, L_D, FT_TRAIN)
FX2t, _ = states_last_and_top(model_r, L_R, FT_TRAIN)
FX9e, _ = states_last_and_top(model_d, L_D, FT_EVAL)
FX2e, F2_top = states_last_and_top(model_r, L_R, FT_EVAL)
print(f"states cached [{time.time()-t0:.0f}s]")

Fmu9, Fmu2, FWr = fit_ridge(FX9t, FX2t)
Fmu9d, Fmu2d = Fmu9.to(DEVICE), Fmu2.to(DEVICE)

# ---- Procrustes: R = U V^T from SVD(A^T B) on centered paired states --------
A_ = (FX9t - Fmu9).double(); B_ = (FX2t - Fmu2).double()
U_, S_, Vh_ = torch.linalg.svd(A_.T @ B_, full_matrices=False)
R = (U_ @ Vh_).float()
AR = A_ @ R.double()
s_opt = float((AR * B_).sum() / (AR * AR).sum())        # exact least-squares scalar
s_norm = float((FX2e - Fmu2).norm(dim=1).mean() / ((FX9e - Fmu9) @ R).norm(dim=1).mean())
ratio = float((FX9e - Fmu9).norm(dim=1).mean() / (FX2e - Fmu2).norm(dim=1).mean())
print(f"semi-orthogonality err {float((R.T@R - torch.eye(D_R)).abs().max()):.2e}")
print(f"optimal scale {s_opt:.4f} | norm-match scale {s_norm:.4f} | donor/recipient norm ratio {ratio:.2f}")
RESULTS["maps"] = dict(procrustes_optimal_scale=round(s_opt, 6),
                       procrustes_normmatch_scale=round(s_norm, 6),
                       donor_over_recipient_norm_ratio=round(ratio, 3),
                       semiorthogonality_err=float((R.T@R - torch.eye(D_R)).abs().max()),
                       dims_discarded=D_D - D_R)
del A_, B_, AR, U_, S_, Vh_; gc.collect()
save()

states cached [439s]
semi-orthogonality err 2.38e-07
optimal scale 0.5555 | norm-match scale 0.7301 | donor/recipient norm ratio 1.43
  [saved exp29_facts_procrustes.json]


In [11]:
# === CELL 9: score every map on the unsolvable bin ==========================
@torch.inference_mode()
def F_first(Wb, idxs, donor_idx=None):
    W, b = Wb; src = donor_idx if donor_idx is not None else idxs
    h = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch); out = []
    try:
        for i in range(0, len(idxs), BATCH):
            sub, ssub = idxs[i:i+BATCH], src[i:i+BATCH]
            _graft["vec"] = (FX9e[ssub].to(DEVICE) - Fmu9d) @ W + b
            ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == FT_EVAL[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        h.remove(); _graft["vec"] = None
    return out

def F_full(Wb, idxs, donor_idx=None):
    W, b = Wb; src = donor_idx if donor_idx is not None else idxs
    items = [FT_EVAL[j] for j in idxs]
    def _vf(i, n): return (FX9e[src[i:i+n]].to(DEVICE) - Fmu9d) @ W + b
    txt = _generate(model_r, items, vec_fn=_vf, layer=L_R)
    return [_match(txt[k], items[k]["answer"]) for k in range(len(items))]

_shuf = list(np.random.default_rng(0).permutation(len(FT_EVAL)))
MAPS = {
    "recon":                  (FWr.to(DEVICE), Fmu2d),
    "procrustes_optscale":    ((R * s_opt).to(DEVICE), Fmu2d),
    "procrustes_normmatch":   ((R * s_norm).to(DEVICE), Fmu2d),
    "procrustes_unscaled":    (R.to(DEVICE), Fmu2d),
}
SC = {"_what": ("All maps scored on the SAME generation-defined unsolvable bin. Native full "
                "is 0 by construction; native first-token is not, and is reported. The task "
                "map is NOT retrained here -- compare against exp27_results.json, which is "
                "valid only if the bin above reproduced n=570.")}
SC["native_first"] = fmt(wilson_bools([F2_top[j] == FT_EVAL[j]["tok"] for j in F_UNSOLV]))
SC["native_full"]  = fmt(wilson_bools([_match(_r_txt[j], FT_EVAL[j]["answer"]) for j in F_UNSOLV]))
print(f"  native            first {SC['native_first']}  full {SC['native_full']}  (full must be 0.000)")
save()

BOOLS = {}
for name, mp in MAPS.items():
    f = F_first(mp, F_UNSOLV); u = F_full(mp, F_UNSOLV)
    BOOLS[name] = (f, u)
    SC[f"{name}_first"] = fmt(wilson_bools(f)); SC[f"{name}_full"] = fmt(wilson_bools(u))
    rc = F.cosine_similarity(((FX9e.to(DEVICE) - Fmu9d) @ mp[0] + mp[1]).cpu(), FX2e, dim=1).numpy()
    SC[f"{name}_recon_cos"] = fmt(bootstrap_ci(rc))
    print(f"  {name:22s} first {SC[f'{name}_first']}  full {SC[f'{name}_full']}")
    RESULTS["scores"] = SC; save()

# specificity: feed each map a MISMATCHED donor state
for name in ["recon", "procrustes_optscale"]:
    SC[f"{name}_shuffled_first"] = fmt(wilson_bools(F_first(MAPS[name], F_UNSOLV, donor_idx=_shuf)))
    print(f"  {name:22s} shuffled-donor first {SC[f'{name}_shuffled_first']}")

# paired tests between the two unsupervised maps
SC["mcnemar_first_recon_vs_procrustes"] = mcnemar_exact(BOOLS["recon"][0], BOOLS["procrustes_optscale"][0])
SC["mcnemar_full_recon_vs_procrustes"]  = mcnemar_exact(BOOLS["recon"][1], BOOLS["procrustes_optscale"][1])
SC["_published_task_map_reference"] = dict(first="0.351", full="0.323",
                                           recon_first="0.326", recon_full="0.342", n=570)
RESULTS["scores"] = SC; save()

  native            first 0.070 [0.052, 0.094]  full 0.000 [0.000, 0.007]  (full must be 0.000)
  [saved exp29_facts_procrustes.json]
  recon                  first 0.324 [0.287, 0.363]  full 0.338 [0.300, 0.378]
  [saved exp29_facts_procrustes.json]
  procrustes_optscale    first 0.299 [0.263, 0.338]  full 0.331 [0.294, 0.371]
  [saved exp29_facts_procrustes.json]
  procrustes_normmatch   first 0.333 [0.295, 0.372]  full 0.368 [0.329, 0.408]
  [saved exp29_facts_procrustes.json]
  procrustes_unscaled    first 0.320 [0.284, 0.360]  full 0.361 [0.322, 0.401]
  [saved exp29_facts_procrustes.json]
  recon                  shuffled-donor first 0.007 [0.003, 0.018]
  procrustes_optscale    shuffled-donor first 0.009 [0.004, 0.020]
  [saved exp29_facts_procrustes.json]


In [12]:
# === CELL 10: summary =======================================================
save()
S = RESULTS.get("scores", {})
b = RESULTS.get("bins", {})
print("\n" + "="*72)
print(f"BIN: n = {b.get('n_unsolvable')}   (EXP27b published 570)")
print(f"     donor {b.get('donor_solve_rate')} (published 0.659) | "
      f"recipient {b.get('recipient_solve_rate')} (published 0.502)")
print("="*72)
print(f"{'map':24s} {'first token':>22s} {'full answer':>22s}")
for n in ["native", "recon", "procrustes_optscale", "procrustes_normmatch", "procrustes_unscaled"]:
    print(f"{n:24s} {S.get(n+'_first','-'):>22s} {S.get(n+'_full','-'):>22s}")
print(f"{'task map (published)':24s} {'0.351':>22s} {'0.323':>22s}")
print("="*72)
print("READ THIS:")
print("  Your paper explains the facts arm by saying naturalistic answers are distinctive")
print("  enough to survive a reconstructive map. Procrustes preserves MORE than the recon")
print("  map, so that explanation predicts procrustes >= recon.")
print("    procrustes close to recon  -> explanation confirmed by a third independent map")
print("    procrustes far above recon -> the explanation needs revising, and you found it first")
print("    procrustes near the floor  -> the dimension reduction is the limit, as on Qwen/Llama")
print(f"\n=== wrote {OUT_JSON} ===")

  [saved exp29_facts_procrustes.json]

BIN: n = 571   (EXP27b published 570)
     donor 0.659 (published 0.659) | recipient 0.5003 (published 0.502)
map                                 first token            full answer
native                     0.070 [0.052, 0.094]   0.000 [0.000, 0.007]
recon                      0.324 [0.287, 0.363]   0.338 [0.300, 0.378]
procrustes_optscale        0.299 [0.263, 0.338]   0.331 [0.294, 0.371]
procrustes_normmatch       0.333 [0.295, 0.372]   0.368 [0.329, 0.408]
procrustes_unscaled        0.320 [0.284, 0.360]   0.361 [0.322, 0.401]
task map (published)                      0.351                  0.323
READ THIS:
  Your paper explains the facts arm by saying naturalistic answers are distinctive
  enough to survive a reconstructive map. Procrustes preserves MORE than the recon
  map, so that explanation predicts procrustes >= recon.
    procrustes close to recon  -> explanation confirmed by a third independent map
    procrustes far above recon -> the

In [13]:
# === CELL 11: CLASS-COUNT SWEEP for the unsupervised maps ===================
# Replicates the design of factual_recall_classcount exactly: the item budget is held
# fixed at N_FIX across every K, so only the number of distinct answer classes varies,
# and each K is scored on its OWN eval subset (bin items whose answer is in the K set).
# The comparable quantity is therefore the DELTA over the reconstruction map, not the
# raw score -- the subsets get easier as K shrinks and raw scores rise for everyone.
#
# The prediction this tests: Procrustes never sees an answer label, so it cannot benefit
# from having fewer classes. Its delta should sit near zero at every K while the task
# map's delta (already measured, +0.213 at K=50) blows up.
SWEEP_JSON = "exp29_facts_classsweep.json"
SWEEP_K    = [50, 200, 800, 3200]
N_FIX      = 4183          # identical to factual_recall_classcount
SWEEP_SEED = 0

from collections import Counter as _Counter
_cnt = _Counter(p["tok"] for p in FT_TRAIN)
_ranked = [t for t, _ in _cnt.most_common()]
print(f"{len(_ranked)} distinct answer classes in train; budget fixed at {N_FIX} items per K")

@torch.inference_mode()
def _first_with_mu(W, b, mu9, idxs):
    """First-token conferral with an explicit centring mean (each K refits its own)."""
    h = model_r.model.layers[L_R].register_forward_hook(patch_vec_batch); out = []
    mu9d_ = mu9.to(DEVICE)
    try:
        for i in range(0, len(idxs), BATCH):
            sub = idxs[i:i+BATCH]
            _graft["vec"] = (FX9e[sub].to(DEVICE) - mu9d_) @ W + b
            ids, m = left_pad([FT_EVAL[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_r(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == FT_EVAL[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        h.remove(); _graft["vec"] = None
    return out

def _fit_procrustes(X9, X2, mu9, mu2):
    A = (X9 - mu9).double(); B = (X2 - mu2).double()
    U, S, Vh = torch.linalg.svd(A.T @ B, full_matrices=False)
    Rk = (U @ Vh).float()
    AR = A @ Rk.double()
    s_opt = float((AR * B).sum() / (AR * AR).sum())
    return Rk, s_opt

SWEEP = {"_what": ("Class-count sweep for the UNSUPERVISED maps, same design as "
                   "factual_recall_classcount: fixed item budget, only class count varies, "
                   "each K scored on its own eval subset. Compare delta_over_recon against "
                   "the task map's delta from that experiment. Procrustes uses no labels, so "
                   "a delta near zero at every K means the class-count effect is about "
                   "supervised learning running out of examples per class, not about the "
                   "representation or about the eval subsets getting easier."),
         "n_fix": N_FIX, "seed": SWEEP_SEED, "n_train_classes_total": len(_ranked),
         "_task_map_reference": {"50": 0.2130, "200": -0.0055, "800": 0.0526, "3200": 0.0369},
         "sweep": []}

for K in SWEEP_K:
    keep = set(_ranked[:K])
    idx = [i for i, p in enumerate(FT_TRAIN) if p["tok"] in keep]
    random.Random(SWEEP_SEED).shuffle(idx)
    idx = idx[:N_FIX]
    ev = [j for j in F_UNSOLV if FT_EVAL[j]["tok"] in keep]
    n_cls = len({FT_TRAIN[i]["tok"] for i in idx})
    if len(ev) < 30 or len(idx) < 200:
        print(f"  K={K}: skipped (eval {len(ev)}, train {len(idx)})"); continue

    mu9k, mu2k, Wrk = fit_ridge(FX9t[idx], FX2t[idx])
    Rk, s_opt_k = _fit_procrustes(FX9t[idx], FX2t[idx], mu9k, mu2k)
    s_norm_k = float((FX2e[ev] - mu2k).norm(dim=1).mean()
                     / ((FX9e[ev] - mu9k) @ Rk).norm(dim=1).mean())
    mu2kd = mu2k.to(DEVICE)

    r_recon = _first_with_mu(Wrk.to(DEVICE), mu2kd, mu9k, ev)
    r_popt  = _first_with_mu((Rk * s_opt_k).to(DEVICE), mu2kd, mu9k, ev)
    r_pnorm = _first_with_mu((Rk * s_norm_k).to(DEVICE), mu2kd, mu9k, ev)
    m_recon, m_popt, m_pnorm = [float(np.mean(x)) for x in (r_recon, r_popt, r_pnorm)]

    row = dict(K=K, train_items=len(idx), train_classes=n_cls,
               examples_per_class=round(len(idx)/max(1, n_cls), 2), eval_n=len(ev),
               recon_first=fmt(wilson_bools(r_recon)),
               procrustes_optscale_first=fmt(wilson_bools(r_popt)),
               procrustes_normmatch_first=fmt(wilson_bools(r_pnorm)),
               delta_procrustes_optscale_minus_recon=round(m_popt - m_recon, 4),
               delta_procrustes_normmatch_minus_recon=round(m_pnorm - m_recon, 4),
               task_map_delta_from_exp27=SWEEP["_task_map_reference"].get(str(K)),
               mcnemar_recon_vs_procrustes_normmatch=mcnemar_exact(r_recon, r_pnorm),
               procrustes_optimal_scale=round(s_opt_k, 6),
               procrustes_normmatch_scale=round(s_norm_k, 6))
    SWEEP["sweep"].append(row)
    print(f"  K={K:>4}  classes {n_cls:>4}  ex/class {row['examples_per_class']:>6}  eval {len(ev):>4}  "
          f"recon {m_recon:.4f}  procrustes {m_pnorm:.4f}  "
          f"delta {m_pnorm-m_recon:+.4f}  (task map delta {row['task_map_delta_from_exp27']})")
    RESULTS["class_sweep"] = SWEEP
    with open(SWEEP_JSON, "w") as f: json.dump(SWEEP, f, indent=2)
    save()

print(f"\n=== wrote {SWEEP_JSON} ===")
print(f"{'K':>5} {'ex/class':>9} {'procrustes delta':>18} {'task map delta':>16}")
for r in SWEEP["sweep"]:
    print(f"{r['K']:>5} {r['examples_per_class']:>9} "
          f"{r['delta_procrustes_normmatch_minus_recon']:>+18.4f} "
          f"{str(r['task_map_delta_from_exp27']):>16}")
print("\nREAD: the task map's delta is +0.213 at K=50 and near zero above it.")
print("  procrustes delta flat near zero  -> the class-count effect is supervised learning")
print("     running out of examples per class. Nothing in the geometry, and not the eval")
print("     subsets getting easier, since procrustes sees the same easier subsets.")
print("  procrustes delta ALSO rises at K=50 -> the easier subset is doing the work and the")
print("     class-count explanation in the paper needs revising.")

7252 distinct answer classes in train; budget fixed at 4183 items per K
  K=  50  classes   50  ex/class  83.66  eval  106  recon 0.3868  procrustes 0.4245  delta +0.0377  (task map delta 0.213)
  [saved exp29_facts_procrustes.json]
  K= 200  classes  200  ex/class  20.91  eval  181  recon 0.3481  procrustes 0.4420  delta +0.0939  (task map delta -0.0055)
  [saved exp29_facts_procrustes.json]
  K= 800  classes  775  ex/class    5.4  eval  281  recon 0.3203  procrustes 0.3559  delta +0.0356  (task map delta 0.0526)
  [saved exp29_facts_procrustes.json]
  K=3200  classes 1893  ex/class   2.21  eval  404  recon 0.2624  procrustes 0.3193  delta +0.0569  (task map delta 0.0369)
  [saved exp29_facts_procrustes.json]

=== wrote exp29_facts_classsweep.json ===
    K  ex/class   procrustes delta   task map delta
   50     83.66            +0.0377            0.213
  200     20.91            +0.0939          -0.0055
  800       5.4            +0.0356           0.0526
 3200      2.21            +0